In [26]:
import numpy as np
import pandas as pd
import mlflow
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv("data/Social_Network_Ads.csv")

In [4]:
X_df = df.iloc[:, :-1]
y_df = df.iloc[:, -1]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.25, random_state=0)

In [6]:
scaler = StandardScaler()
scaler.fit(X_train, y_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(pd.DataFrame({
    "mean (unscaled)": X_train.mean(),
    "std (unscaled)": X_train.std(),
    "mean (scaled)": X_train_scaled.mean(),
    "std (scaled)": X_train_scaled.std(),
}))

                 mean (unscaled)  std (unscaled)  mean (scaled)  std (scaled)
Age                    38.126667       10.114592   1.495100e-16           1.0
EstimatedSalary     69583.333333    34548.541619   1.495100e-16           1.0


# Train and register a model

In [ ]:
from mlflow.models import infer_signature

class SklearnModelWrapper(mlflow.pyfunc.PythonModel):
  def __init__(self, model):
    self.model = model

  def predict(self, context, model_input, params=None):
    return self.model.predict(model_input)

run_name = 'simple_random_forest'

with mlflow.start_run(run_name=run_name):
    max_depth = 4
    classifier = RandomForestClassifier(random_state=0, n_estimators=10, max_depth=max_depth)
    classifier.fit(X_train_scaled, y_train)

    mlflow.log_param("max_depth", max_depth)

    y_pred = classifier.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("recall", recall_score(y_test, y_pred))
    mlflow.log_metric("f1_score", f1_score(y_test, y_pred))

    wrappedModel = SklearnModelWrapper(classifier)

    signature = infer_signature(X_train_scaled, wrappedModel.predict(None, X_train_scaled))

    mlflow.sklearn.log_model(
        sk_model=classifier,
        name="random_forest_model",
        signature=signature,
        skops_trusted_types=[
            "sklearn.tree._tree.Tree"
        ],
    )

In [40]:
run_id = mlflow.search_runs(filter_string=f'tags.mlflow.runName = "{run_name}"').iloc[0].run_id
run_id

'e1b5c8fc56f44d0286fb2d694c1de6ea'

In [41]:
model_name = "random_forest"
model_version = mlflow.register_model(f"runs:/{run_id}/random_forest_model", model_name)

# Registering the model takes a few seconds, so add a small delay
time.sleep(15)

Registered model 'random_forest' already exists. Creating a new version of this model...
2026/10/18 21:48:40 WARNING mlflow.tracking._model_registry.fluent: Run with id e1b5c8fc56f44d0286fb2d694c1de6ea has no artifacts at artifact path 'random_forest_model', registering model based on models:/m-9b395f70646c482db5888a9d0bf93113 instead
Created version '2' of model 'random_forest'.


# K-Fold CV Experiment Tracking with MLFlow

In [42]:
k_values = [3, 5, 7, 10, 15, 20]

results = []

# Load the registered model
model = mlflow.sklearn.load_model(
    f"models:/{model_name}/None"
)

mlflow.set_experiment("kfold_experiment")

with mlflow.start_run(run_name="kfold_comparison") as parent_run:
  # Record information shared by all configurations
  mlflow.log_param("model", type(classifier).__name__)
  mlflow.log_param("k_values", str(k_values))
  mlflow.log_param("scoring", "accuracy")

  for k in k_values:
      with mlflow.start_run(
          run_name=f"k_{k}",
          nested=True
      ):
          scores = cross_val_score(
              model,
              X_train,
              y_train,
              scoring="accuracy",
              cv=k
          )

          stats = {
              "k": k,
              "mean": np.mean(scores),
              "std": np.std(scores),
              "min": np.min(scores),
              "max": np.max(scores),
          }

          # Log parameters and aggregate metrics
          mlflow.log_param("cv_folds", k)

          for metric in ["mean", "std", "min", "max"]:
              mlflow.log_metric(
                  metric,
                  float(stats[metric])
              )

          # Log each fold's accuracy separately
          for fold, score in enumerate(scores, start=1):
              mlflow.log_metric(
                  f"fold_{fold}_accuracy",
                  float(score)
              )

          results.append(stats)